In [229]:
import pandas as pd
import json

In [230]:
df_terms_final = pd.read_parquet('cleaned_aws_terms.parquet')
df_products = pd.read_parquet('cleaned_aws_products.parquet')

In [231]:
print("Terms shape:", df_terms_final.shape)
print("Products shape:", df_products.shape)
df_products.head(3)

Terms shape: (61473, 5)
Products shape: (100000, 79)


,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.currentGeneration,attributes.instanceFamily,attributes.vcpu,attributes.physicalProcessor,attributes.clockSpeed,attributes.memory,attributes.storage,attributes.networkPerformance,attributes.processorArchitecture,attributes.tenancy,attributes.operatingSystem,attributes.licenseModel,attributes.usagetype,attributes.operation,attributes.availabilityzone,attributes.capacitystatus,attributes.classicnetworkingsupport,attributes.dedicatedEbsThroughput,attributes.dedicatedEbsThroughputDescription,attributes.ecu,attributes.enhancedNetworkingSupported,attributes.gpuMemory,attributes.instanceFamilyCategory,attributes.instancesku,attributes.intelAvxAvailable,attributes.intelAvx2Available,attributes.intelTurboAvailable,attributes.marketoption,attributes.normalizationSizeFactor,attributes.preInstalledSw,attributes.regionCode,attributes.servicename,attributes.vpcnetworkingsupport,attributes.processorFeatures,attributes.gpu,attributes.instanceCapacity-12xlarge,attributes.instanceCapacity-16xlarge,attributes.instanceCapacity-24xlarge,attributes.instanceCapacity-2xlarge,attributes.instanceCapacity-32xlarge,attributes.instanceCapacity-4xlarge,attributes.instanceCapacity-8xlarge,attributes.instanceCapacity-Large,attributes.instanceCapacity-Xlarge,attributes.physicalCores,attributes.group,attributes.groupDescription,attributes.transferType,attributes.fromLocation,attributes.fromLocationType,attributes.toLocation,attributes.toLocationType,attributes.fromRegionCode,attributes.toRegionCode,attributes.instanceCapacity-18xlarge,attributes.instanceCapacity-9xlarge,attributes.resourceType,attributes.productType,attributes.provisioned,attributes.volumeApiName,attributes.instanceCapacity-Metal,attributes.instanceCapacity-Medium,attributes.storageMedia,attributes.volumeType,attributes.maxVolumeSize,attributes.maxIopsvolume,attributes.maxIopsBurstPerformance,attributes.maxThroughputvolume,attributes.instance,attributes.snapshotarchivefeetype,attributes.ebsOptimized,attributes.elasticGraphicsType,attributes.instanceCapacity-10xlarge
0,RSH2Y67N4H4CFBQ4,Compute Instance,AmazonEC2,Asia Pacific (Malaysia),AWS Region,c7gd.medium,Yes,Compute optimized,1,AWS Graviton3 Processor,2.5 GHz,2 GiB,1 x 59 NVMe SSD,Up to 12500 Megabit,64-bit,Shared,RHEL,No License required,APS7-Reservation:c7gd.medium,RunInstances:0010,NA,AllocatedCapacityReservation,false,Up to 10000 Mbps,315 Mbps,NA,Yes,NA,Compute Optimized,7CQ8E9D98XGDXC9R,No,No,No,OnDemand,2,NA,ap-southeast-5,Amazon Elastic Compute Cloud,true,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,YHBHTMRVRUX8M33Y,Compute Instance,AmazonEC2,Asia Pacific (Malaysia),AWS Region,m7i.large,Yes,General purpose,2,Intel Xeon Scalable (Sapphire Rapids),3.2 GHz,8 GiB,EBS only,Up to 12500 Megabit,64-bit,Shared,Red Hat Enterprise Linux with HA,No License required,APS7-Reservation:m7i.large,RunInstances:1010,NA,AllocatedCapacityReservation,false,Up to 10000 Mbps,650 Mbps,NA,Yes,NA,General Purpose,SS7SJ8ZYRUB9AY5C,Yes,Yes,Yes,OnDemand,4,NA,ap-southeast-5,Amazon Elastic Compute Cloud,true,Intel AVX; Intel AVX2; Intel AVX512; Intel Turbo; Intel AMX,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,JSEGQBVEG2Y4NYJR,Compute Instance,AmazonEC2,Asia Pacific (Thailand),AWS Region,c6in.24xlarge,Yes,Compute optimized,96,Intel Xeon 8375C (Ice Lake),3.5 GHz,192 GiB,EBS only,150000 Megabit,64-bit,Host,Linux,No License required,APS9-HostBoxUsage:c6in.24xlarge,RunInstances:0100,NA,Used,false,75000 Mbps,75000 Mbps,NA,Yes,NA,Compute Optimized,None,Yes,Yes,Yes,OnDemand,192,SQL Ent,ap-southeast-7,Amazon Elastic Compute Cloud,true,Intel AVX; Intel 

In [232]:
if 'attributes.marketoption' in df_products.columns:
    df_products.drop(columns=['attributes.marketoption'], errors='ignore', inplace=True)

print (f"Dimensions (rows, cols): {df_products.shape}")

Dimensions (rows, cols): (100000, 78)


In [233]:
if 'attributes.servicename' in df_products.columns:
    df_products.drop(columns=['attributes.servicename'], errors='ignore', inplace=True)

print (f"Dimensions (rows, cols): {df_products.shape}")

Dimensions (rows, cols): (100000, 77)


Βάση της οικογένειας στην οποία ανήκουν, διαιρώ τις υπηρεσίες του EC2 σε 3 μεγάλες οικογένειες. Το ec2 μέσα περιέχει έτοιμα instances προς χρήση με μορφή virtualization, έτοιμα instances τα οποία όμως παίρνουμε όλο το σίδερο, και "εργαλεία" προκειμένου κάποιος να φτιάξει το instance όπως θέλει ο ιδιος

In [234]:
df_compute = df_products[df_products['productFamily'] == 'Compute Instance'].reset_index(drop=True)

df_bare_metal = df_products[df_products['productFamily'] == 'Compute Instance (bare metal)'].reset_index(drop=True)

df_extras = df_products[~df_products['productFamily'].isin(['Compute Instance', 'Compute Instance (bare metal)'])].reset_index(drop=True)

#### Ανάλυση Compute instance

In [235]:
# attributes.regionCode
# Πεδία προδιορισμού (που, πότε, κατηγορία, περιοχήσ, ποιος)
id_columns = ['sku', 'productFamily', 
              'attributes.servicecode', 
              'attributes.location', 
              'attributes.locationType']

In [236]:
#Πεδία χαρακτηρισμού
feature_columns = ['attributes.instanceType', 'attributes.instanceFamily',
                   'attributes.instanceFamilyCategory', 'attributes.vcpu',
                   'attributes.memory', 'attributes.operatingSystem',
                   'attributes.tenancy', 'attributes.processorArchitecture',
                   'attributes.physicalProcessor', 'attributes.clockSpeed',
                   'attributes.storage']

In [237]:
total = []
total.extend(id_columns)
total.extend(feature_columns)
df_compute[total].head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.instanceFamily,attributes.instanceFamilyCategory,attributes.vcpu,attributes.memory,attributes.operatingSystem,attributes.tenancy,attributes.processorArchitecture,attributes.physicalProcessor,attributes.clockSpeed,attributes.storage
0,RSH2Y67N4H4CFBQ4,Compute Instance,AmazonEC2,Asia Pacific (Malaysia),AWS Region,c7gd.medium,Compute optimized,Compute Optimized,1,2 GiB,RHEL,Shared,64-bit,AWS Graviton3 Processor,2.5 GHz,1 x 59 NVMe SSD
1,YHBHTMRVRUX8M33Y,Compute Instance,AmazonEC2,Asia Pacific (Malaysia),AWS Region,m7i.large,General purpose,General Purpose,2,8 GiB,Red Hat Enterprise Linux with HA,Shared,64-bit,Intel Xeon Scalable (Sapphire Rapids),3.2 GHz,EBS only
2,JSEGQBVEG2Y4NYJR,Compute Instance,AmazonEC2,Asia Pacific (Thailand),AWS Region,c6in.24xlarge,Compute optimized,Compute Optimized,96,192 GiB,Linux,Host,64-bit,Intel Xeon 8375C (Ice Lake),3.5 GHz,EBS only
3,646B9QM23GWYVTAK,Compute Instance,AmazonEC2,Africa (Cape Town),AWS Region,c7i.xlarge,Compute optimized,Compute Optimized,4,8 GiB,Red Hat Enterprise Linux with HA,Shared,64-bit,Intel Xeon Scalable (Sapphire Rapids),3.2 GHz,EBS only
4,SSMSW7PJQYNKBTE4,Compute Instance,AmazonEC2,AWS GovCloud (US-West),AWS Region,m6i.large,General purpose,General Purpose,2,8 GiB,RHEL,Shared,64-bit,Intel Xeon 8375C (Ice Lake),3.5 GHz,EBS only


In [238]:
# Βρίσκουμε ποιες στήλες του DataFrame ΔΕΝ περιέχονται στη λίστα 'total'. Η total περιέχει όλα τα στοιχεία που θέλω να κρατήσω ως βασικά
remaining_cols = [col for col in df_compute.columns if col not in total]

#Δημιουργώ αντίγραφο του compute με τις στήλες που θέλω
df_final_compute = df_compute[total].copy()

#Φτιάχνω μία νέα στήλη με τις υπολοιπόμενες στήλες σε μορφή λεξικού
df_final_compute['additionalAttributes'] = df_compute[remaining_cols].apply(
    lambda row: json.dumps({k: v for k, v in row.to_dict().items() if pd.notna(v)}), 
    axis=1
)

In [239]:
#κάνω merge τον πίνακα compute με τον πίνακα των τιμών βάση του sku. Επειδή έιχα 2 στήλες sku με το ίδιο όνομα η μία φεύγει. Για να μπει ένα πεδίο
#, στον τελικό πίνακακ πρέπει να υπάρχει και στους 2 το sku. Ουσιαστικά παίρνει γραμμή 1 τουc compute, ψάνψει όλο το terms για το sku αυτό, και αν το
# βρει φτιάνει νέα εγγραφή στον πίνακα master, αλλιώς το αφήνει εκτός. Και συνεχίζει. Δεν ανησυχώ για διπλότυπα. Βάση του καθαρισμού δεν υπάρχουν, αλλά και να υπάρχουν τα διαχειρίζεται η εντολή
df_master_compute = pd.merge(
    df_final_compute, 
    df_terms_final, 
    on='sku', 
    how='inner'
)
df_master_compute = df_master_compute.reset_index(drop=True)

In [240]:
pd.set_option('display.max_colwidth', 50)
df_master_compute.head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.instanceFamily,attributes.instanceFamilyCategory,attributes.vcpu,attributes.memory,attributes.operatingSystem,attributes.tenancy,attributes.processorArchitecture,attributes.physicalProcessor,attributes.clockSpeed,attributes.storage,additionalAttributes,rateCode,description,unit,priceUSD
0,646B9QM23GWYVTAK,Compute Instance,AmazonEC2,Africa (Cape Town),AWS Region,c7i.xlarge,Compute optimized,Compute Optimized,4,8 GiB,Red Hat Enterprise Linux with HA,Shared,64-bit,Intel Xeon Scalable (Sapphire Rapids),3.2 GHz,EBS only,"{""attributes.currentGeneration"": ""Yes"", ""attri...",646B9QM23GWYVTAK.JRTCKXETXF.6YS6EN2CT7,$1.867 per Unused Reservation RHEL with HA and...,Hrs,1.86700
1,SSMSW7PJQYNKBTE4,Compute Instance,AmazonEC2,AWS GovCloud (US-West),AWS Region,m6i.large,General purpose,General Purpose,2,8 GiB,RHEL,Shared,64-bit,Intel Xeon 8375C (Ice Lake),3.5 GHz,EBS only,"{""attributes.currentGeneration"": ""Yes"", ""attri...",SSMSW7PJQYNKBTE4.JRTCKXETXF.6YS6EN2CT7,$0.2174 per Unused Reservation RHEL with SQL W...,Hrs,0.21740
2,QATMMBSN25PFKEFA,Compute Instance,AmazonEC2,Europe (Zurich),AWS Region,c6in.4xlarge,Compute optimized,Compute Optimized,16,32 GiB,RHEL,Dedicated,64-bit,Intel Xeon 8375C (Ice Lake),3.5 GHz,EBS only,"{""attributes.currentGeneration"": ""Yes"", ""attri...",QATMMBSN25PFKEFA.JRTCKXETXF.6YS6EN2CT7,$3.34504 per Dedicated Unused Reservation RHEL...,Hrs,3.34504
3,X44S6FJV8TJ3R88D,Compute Instance,AmazonEC2,US East (Ohio),AWS Region,i7i.8xlarge,Storage optimized,Storage Optimized,32,256 GiB,Ubuntu Pro,Shared,64-bit,Intel Xeon Scalable (Emerald Rapids),3.2 GHz,2 x 3750 NVMe SSD,"{""attributes.currentGeneration"": ""Yes"", ""attri...",X44S6FJV8TJ3R88D.JRTCKXETXF.6YS6EN2CT7,$3.0762 per On Demand Ubuntu Pro i7i.8xlarge I...,Hrs,3.07620
4,6P49AUTUVZ7E2G82,Compute Instance,AmazonEC2,AWS GovCloud (US-West),AWS Region,m7i-flex.12xlarge,General purpose,General Purpose,48,192 GiB,Ubuntu Pro,Shared,64-bit,Intel Xeon Scalable (Sapphire Rapids),3.5 GHz,EBS only,"{""attributes.currentGeneration"": ""Yes"", ""attri...",6P49AUTUVZ7E2G82.JRTCKXETXF.6YS6EN2CT7,$2.9808 per Unused Reservation Ubuntu Pro m7i-...,Hrs,2.98080


#### Ανάλυση Compute instance (Bare metal)

In [241]:
id_columns_metal = ['sku','productFamily', 
                    'attributes.servicecode', 
                    'attributes.location', 
                    'attributes.locationType']

In [242]:
feature_columns_metal = [ 'attributes.instanceType', 'attributes.instanceFamily',
                        'attributes.instanceFamilyCategory', 'attributes.vcpu',
                        'attributes.memory', 'attributes.operatingSystem',
                        'attributes.tenancy', 'attributes.processorArchitecture',
                        'attributes.physicalProcessor', 'attributes.clockSpeed',
                        'attributes.storage',
                        'attributes.networkPerformance','attributes.dedicatedEbsThroughput',
]

# 3. Συνδυάζουμε τα total για το copy
total_metal = id_columns_metal + feature_columns_metal

In [243]:
total_metal = []
total_metal.extend(id_columns_metal)
total_metal.extend(feature_columns_metal)
df_bare_metal[total].head()


,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.instanceFamily,attributes.instanceFamilyCategory,attributes.vcpu,attributes.memory,attributes.operatingSystem,attributes.tenancy,attributes.processorArchitecture,attributes.physicalProcessor,attributes.clockSpeed,attributes.storage
0,XP2N78YHZ3RHY6XC,Compute Instance (bare metal),AmazonEC2,Asia Pacific (Hong Kong),AWS Region,m7i.metal-24xl,General purpose,General Purpose,96,384 GiB,SUSE,Dedicated,64-bit,Intel Xeon Scalable (Sapphire Rapids),3.2 GHz,EBS only
1,RDXTJ6FD7FCXUMVM,Compute Instance (bare metal),AmazonEC2,EU (Paris),AWS Region,r8g.metal-24xl,Memory optimized,Memory Optimized,96,768 GiB,Linux,Dedicated,64-bit,AWS Graviton4 Processor,2.7 GHz,EBS only
2,EXWP5EWE78BBB99C,Compute Instance (bare metal),AmazonEC2,Asia Pacific (Tokyo),AWS Region,i7i.metal-24xl,Storage optimized,Storage Optimized,96,768 GiB,Windows,Dedicated,64-bit,Intel Xeon Scalable (Emerald Rapids),3.2 GHz,6 x 3750 NVMe SSD
3,9NNG2H438M6C2QWA,Compute Instance (bare metal),AmazonEC2,Asia Pacific (Tokyo),AWS Region,c5.metal,Compute optimized,Compute Optimized,96,192 GiB,Linux,Dedicated,64-bit,Intel Xeon Platinum 8275L,3.4 GHz,EBS only
4,S4ERBHDBT9SV8EUK,Compute Instance (bare metal),AmazonEC2,Asia Pacific (Hyderabad),AWS Region,i3en.metal,Storage optimized,Storage Optimized,96,768 GiB,SUSE,Host,64-bit,Intel Xeon Platinum 8175,3.1 GHz,8 x 7500 NVMe SSD


In [244]:
remaining_cols_metal = [col for col in df_bare_metal.columns if col not in total_metal]
df_final_metal = df_bare_metal[total_metal].copy()

df_final_metal['additionalAttributes'] = df_bare_metal[remaining_cols_metal].apply(
    lambda row: json.dumps({k: v for k, v in row.to_dict().items() if pd.notna(v)}), 
    axis=1
)

In [245]:
df_master_metal = pd.merge(
    df_final_metal, 
    df_terms_final, 
    on='sku', 
    how='inner'
)

df_master_metal = df_master_metal.reset_index(drop=True)

In [246]:
pd.set_option('display.max_colwidth', 50)
df_master_metal.head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.instanceFamily,attributes.instanceFamilyCategory,attributes.vcpu,attributes.memory,attributes.operatingSystem,attributes.tenancy,attributes.processorArchitecture,attributes.physicalProcessor,attributes.clockSpeed,attributes.storage,attributes.networkPerformance,attributes.dedicatedEbsThroughput,additionalAttributes,rateCode,description,unit,priceUSD
0,XP2N78YHZ3RHY6XC,Compute Instance (bare metal),AmazonEC2,Asia Pacific (Hong Kong),AWS Region,m7i.metal-24xl,General purpose,General Purpose,96,384 GiB,SUSE,Dedicated,64-bit,Intel Xeon Scalable (Sapphire Rapids),3.2 GHz,EBS only,37500 Megabit,30000 Mbps,"{""attributes.currentGeneration"": ""Yes"", ""attri...",XP2N78YHZ3RHY6XC.JRTCKXETXF.6YS6EN2CT7,$7.44308 per Dedicated SUSE m7i.metal-24xl Ins...,Hrs,7.44308
1,RDXTJ6FD7FCXUMVM,Compute Instance (bare metal),AmazonEC2,EU (Paris),AWS Region,r8g.metal-24xl,Memory optimized,Memory Optimized,96,768 GiB,Linux,Dedicated,64-bit,AWS Graviton4 Processor,2.7 GHz,EBS only,40 Gigabit,30000 Mbps,"{""attributes.currentGeneration"": ""Yes"", ""attri...",RDXTJ6FD7FCXUMVM.JRTCKXETXF.6YS6EN2CT7,$7.30646 per Dedicated Linux r8g.metal-24xl In...,Hrs,7.30646
2,9NNG2H438M6C2QWA,Compute Instance (bare metal),AmazonEC2,Asia Pacific (Tokyo),AWS Region,c5.metal,Compute optimized,Compute Optimized,96,192 GiB,Linux,Dedicated,64-bit,Intel Xeon Platinum 8275L,3.4 GHz,EBS only,25 Gigabit,14000 Mbps,"{""attributes.currentGeneration"": ""Yes"", ""attri...",9NNG2H438M6C2QWA.JRTCKXETXF.6YS6EN2CT7,$6.758 per Dedicated Unused Reservation Linux ...,Hrs,6.75800
3,E4AHGTJMQH6EJSZ3,Compute Instance (bare metal),AmazonEC2,Middle East (UAE),AWS Region,i7i.metal-24xl,Storage optimized,Storage Optimized,96,768 GiB,Windows,Dedicated,64-bit,Intel Xeon Scalable (Emerald Rapids),3.2 GHz,6 x 3750 NVMe SSD,Up to 56.25 Gigabit,30 Gbps,"{""attributes.currentGeneration"": ""Yes"", ""attri...",E4AHGTJMQH6EJSZ3.JRTCKXETXF.6YS6EN2CT7,$11.659 per Dedicated Windows BYOL i7i.metal-2...,Hrs,11.65900
4,STQK5XRMEGSJRNQX,Compute Instance (bare metal),AmazonEC2,EU (London),AWS Region,x2idn.metal,Memory optimized,Memory Optimized,128,2048 GiB,Windows,Dedicated,64-bit,Intel Xeon Scalable (Icelake),3.5 GHz,2 x 1900 NVMe SSD,100 Gigabit,80 Gbps,"{""attributes.currentGeneration"": ""Yes"", ""attri...",STQK5XRMEGSJRNQX.JRTCKXETXF.6YS6EN2CT7,$24.8572 per Dedicated Unused Reservation Wind...,Hrs,24.85720


#### Aνάλυση υπολοίπων

In [247]:
id_columns_extras = ['sku', 'productFamily', 
                    'attributes.servicecode', 
                    'attributes.location', 
                    'attributes.locationType']   


In [248]:
remaining_cols_extras = [col for col in df_extras.columns if col not in id_columns_extras]

df_final_extras = df_extras[id_columns_extras].copy()

# Επιπλέον στήλη με όλες τις όχι βασικές στήλες. Πετάμε NAN 
df_final_extras['additional_attributes'] = df_extras[remaining_cols_extras].apply(
    lambda row: json.dumps({k: v for k, v in row.to_dict().items() if pd.notna(v)}), 
    axis=1
)

In [249]:
df_master_extras = pd.merge(
    df_final_extras, 
    df_terms_final, 
    on='sku', 
    how='inner'
)

df_master_extras = df_master_extras.reset_index(drop=True)


In [250]:
df_master_extras.head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,additional_attributes,rateCode,description,unit,priceUSD
0,T3YNM5Q6ZW8SZ9SX,IP Address,AmazonEC2,US East (Verizon) - Houston,AWS Wavelength Zone,"{""attributes.usagetype"": ""USE1WL1IAH1-CarrierI...",T3YNM5Q6ZW8SZ9SX.JRTCKXETXF.6YS6EN2CT7,$0.005 per hour for CarrierIP:AdditionalAddres...,hour,0.005
1,83K59X2AV5V696PA,Data Transfer,AmazonEC2,None,None,"{""attributes.usagetype"": ""DataTransfer-Regiona...",83K59X2AV5V696PA.JRTCKXETXF.6YS6EN2CT7,$0.010 per GB Regional Data Transfer - in/out/...,GB,0.010
2,FEXHXKG7U556TTZB,Data Transfer,AmazonEC2,None,None,"{""attributes.usagetype"": ""APN1-APN2-AWS-Out-By...",FEXHXKG7U556TTZB.JRTCKXETXF.6YS6EN2CT7,$0.09 per GB - Asia Pacific (Tokyo) data trans...,GB,0.090
3,ACP577JRFQBDA2YU,Dedicated Host,AmazonEC2,Europe (Spain),AWS Region,"{""attributes.instanceType"": ""c6in"", ""attribute...",ACP577JRFQBDA2YU.JRTCKXETXF.6YS6EN2CT7,$9.011 per On Demand C6IN Dedicated Host Hour,Hrs,9.011
4,UPPFJF6RZDMQQNTN,Fee,AmazonEC2,US East (Verizon) - New York,AWS Wavelength Zone,"{""attributes.usagetype"": ""USE1WL1NYC1-Dedicate...",UPPFJF6RZDMQQNTN.JRTCKXETXF.6YS6EN2CT7,$2.00 once per hour when you're running at lea...,Hrs,2.000


In [251]:
# summary_data = []
# for col in df_extras.columns:
#     sample_vals = list(df_extras[col].dropna().unique()[:5])
#     summary_data.append({'Column': col, 'Sample_Values': sample_vals})

# # Δημιουργία ενός συνοπτικού DataFrame
# df_summary = pd.DataFrame(summary_data)

# # Εμφάνιση χωρίς περικοπές
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_colwidth', None)
# df_summary

In [ ]:
# print("Duplicates", df_final_metal['sku'].duplicated().sum())

# print("Duplicates", df_terms_final['sku'].duplicated().sum())